Setup/installations

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

!pip -q install folium pyproj scikit-learn

import pandas as pd
import numpy as np
import folium
from sklearn.neighbors import BallTree
from pyproj import Transformer
from math import sin, cos, asin, sqrt
from IPython.display import display

Load in all threshold tables

In [ ]:
night_22_5 = pd.read_csv("/content/drive/MyDrive/night_threshold_tables/night_22_5_csv")
night_30   = pd.read_csv("/content/drive/MyDrive/night_threshold_tables/night_30_0_csv")
night_37_5 = pd.read_csv("/content/drive/MyDrive/night_threshold_tables/night_37_5_csv")

configurations and paramaters

In [ ]:
# --- FILE PATHS (adjust if needed) ---
STREETLIGHT_CSV = "/content/drive/My Drive/Street_Lights.csv"
CRASH_CSV       = "/content/drive/My Drive/Crashes_in_DC.csv"

# --- PARAMETERS ---
START_DATE = "2020-01-01"
END_DATE   = "2025-04-30"
NIGHT_HOURS = list(range(0, 6)) + list(range(20, 24))

THRESHOLD_M = 30   # use 22.5, 30.0, 37.5

WITHIN_THRESHOLD_M = THRESHOLD_M
FAR_THRESHOLD_M    = THRESHOLD_M
CLUSTER_RADIUS_M   = THRESHOLD_M
MIN_CLUSTER_POINTS    = 5
DRAW_SIZE_CIRCLES     = True
RADIUS_TOLERANCE_M    = 1e-6

EARTH_RADIUS_M = 6_371_000.0

add helpers/check longitude and latitude

In [ ]:
def looks_projected(df, lat_col="LATITUDE", lon_col="LONGITUDE"):
    lat_max = df[lat_col].abs().max()
    lon_max = df[lon_col].abs().max()
    return (lat_max > 90) or (lon_max > 180) or (lat_max > 1000) or (lon_max > 1000)


Lode and clean streetlights

Load raw data
→ Read the streetlight CSV into a pandas DataFrame.

Normalize column names
→ Strip spaces and uppercase all headers for consistency.

 Standardize coordinate columns
→ Rename X and Y to LONGITUDE and LATITUDE if needed.

 Drop invalid rows
→ Remove any rows missing LATITUDE or LONGITUDE.

 Ensure numeric coordinates
→ Convert coordinates to floats; drop anything unconvertible.

 Detect if data is projected (in meters)
→ Use looks_projected() to check if coordinates need conversion.

 Reproject if needed (EPSG:3857 → EPSG:4326)
→ Transform projected meters into standard lat/lon (WGS84).

 Remove duplicates
→ Drop duplicate lat/lon points to prevent map clutter and analysis bias.

 Final sanity check
→ Raise an error if the dataset ends up empty after cleaning.

In [ ]:
from pyproj import Transformer
import numpy as np

streetlights = pd.read_csv(STREETLIGHT_CSV, low_memory=False)

# Normalize column names
orig_cols = streetlights.columns.tolist()
streetlights.rename(columns={c: c.strip().upper() for c in streetlights.columns}, inplace=True)
print("Columns (after normalize):\n", streetlights.columns.tolist())
if orig_cols != streetlights.columns.tolist():
    print("⚙️ Renamed some columns to upper/trimmed whitespace.")

# --- Helpers -----------------------------------------------------------------
def has_latlon(df):
    return {"LAT", "LON"}.issubset(df.columns) or {"LATITUDE", "LONGITUDE"}.issubset(df.columns)

def detect_xy_crs(x, y):
    """
    Heuristically detect the CRS of projected X/Y columns.

    Returns: ("EPSG:xxxx", reason_string)
    """
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    # Drop NaNs for stats
    xs = x[np.isfinite(x)]
    ys = y[np.isfinite(y)]
    if xs.empty or ys.empty:
        return None, "insufficient finite X/Y values"

    xmin, xmax = xs.min(), xs.max()
    ymin, ymax = ys.min(), ys.max()

    # If values look like degrees (already lon/lat)
    if (-180 <= xmin <= 180) and (-180 <= xmax <= 180) and (-90 <= ymin <= 90) and (-90 <= ymax <= 90):
        return "EPSG:4326", "values appear to be degrees"

    # Web Mercator meters range ~ ±20,037,508
    if all(abs(v) <= 2.1e7 for v in [xmin, xmax, ymin, ymax]):
        # Typical DC Web Mercator should be around x ~ -8.6e6, y ~ 4.6e6
        return "EPSG:3857", "values within Web Mercator meter range"

    # Maryland StatePlane (NAD83) US-ft (DC commonly uses 2248)
    # Typical magnitudes in hundreds of thousands to a few million feet
    if all(1e4 <= abs(v) <= 1e7 for v in [xmin, xmax, ymin, ymax]):
        return "EPSG:2248", "values look like StatePlane Maryland (US-ft)"

    # NAD83 / Maryland meters variant (less common for DDOT tabular): EPSG:26985
    if all(1e3 <= abs(v) <= 1e6 for v in [xmin, xmax, ymin, ymax]):
        return "EPSG:26985", "values look like StatePlane Maryland (meters)"

    return None, "unable to infer CRS from value ranges"

def transform_xy_to_wgs84(x, y, src_epsg):
    tf = Transformer.from_crs(src_epsg, "EPSG:4326", always_xy=True)
    lon, lat = tf.transform(x.to_numpy(), y.to_numpy())
    return pd.Series(lat), pd.Series(lon)

# --- Coordinate column handling ---------------------------------------------
# Prefer existing LAT/LON if present
if has_latlon(streetlights):
    # Normalize to LATITUDE/LONGITUDE names
    if {"LAT", "LON"}.issubset(streetlights.columns):
        streetlights.rename(columns={"LAT": "LATITUDE", "LON": "LONGITUDE"}, inplace=True)
    # If already LATITUDE/LONGITUDE, leave as-is
    print("✅ Using existing LATITUDE/LONGITUDE columns.")

elif {"X", "Y"}.issubset(streetlights.columns):
    # Try to detect CRS of X/Y
    src_epsg, reason = detect_xy_crs(streetlights["X"], streetlights["Y"])
    if src_epsg is None:
        raise ValueError(f"❌ Could not infer CRS for X/Y — {reason}. Provide the correct EPSG code.")

    print(f"↔️ Converting X/Y → WGS84 (EPSG:4326) from {src_epsg} ({reason}).")

    # Transform
    lat_wgs, lon_wgs = transform_xy_to_wgs84(streetlights["X"], streetlights["Y"], src_epsg)
    streetlights["LATITUDE"] = pd.to_numeric(lat_wgs, errors="coerce")
    streetlights["LONGITUDE"] = pd.to_numeric(lon_wgs, errors="coerce")

else:
    raise ValueError("❌ Could not find coordinate columns (LAT/LON or X/Y).")

# --- Diagnostics: show a few sample coordinates -----------------------------
print("\n=== STREETLIGHT COORDINATE SAMPLES ===")
print(streetlights[["LATITUDE", "LONGITUDE"]].head(5))

# --- Clean up ----------------------------------------------------------------
# Drop bad coords
streetlights = streetlights.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
streetlights["LATITUDE"]  = pd.to_numeric(streetlights["LATITUDE"], errors="coerce")
streetlights["LONGITUDE"] = pd.to_numeric(streetlights["LONGITUDE"], errors="coerce")
streetlights = streetlights.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

# Fix common sign issues: DC longitudes must be negative (~ -77)
if streetlights["LONGITUDE"].median() > 0:
    print("🧭 Detected positive longitudes; flipping sign to negative (W hemisphere).")
    streetlights["LONGITUDE"] = -streetlights["LONGITUDE"].abs()

# Remove exact duplicate points
streetlights = streetlights.drop_duplicates(subset=["LATITUDE","LONGITUDE"]).reset_index(drop=True)

# Only keep DDOT lights that still physically exist (if available)
if "ASSETSTATUS" in streetlights.columns:
    pre_status = len(streetlights)
    streetlights = streetlights[
        ~streetlights["ASSETSTATUS"].astype(str).str.contains("Removed|Decommissioned", case=False, na=False)
    ].copy()
    print(f"Filtered removed/decommissioned assets: {pre_status - len(streetlights)} dropped.")

# Clip to central DC extent
LAT_MIN, LAT_MAX = 38.81, 38.995
LON_MIN, LON_MAX = -77.12, -76.91

pre_bbox = len(streetlights)
streetlights = streetlights[
    (streetlights["LATITUDE"].between(LAT_MIN, LAT_MAX)) &
    (streetlights["LONGITUDE"].between(LON_MIN, LON_MAX))
].copy()
print(f"🗺️ Filtered streetlights to {len(streetlights):,} within DC bounds (from {pre_bbox:,}).")

# Final sanity checks
if streetlights.empty:
    raise ValueError("❌ Streetlight dataset is empty after filtering — check coordinate range or filters.")

print("\n=== STREETLIGHT COORDINATE CHECK ===")
print("Latitude range:", streetlights["LATITUDE"].min(), "→", streetlights["LATITUDE"].max())
print("Longitude range:", streetlights["LONGITUDE"].min(), "→", streetlights["LONGITUDE"].max())
print("Total streetlights:", len(streetlights))
print("✅ Ready for analysis.")




Lode and prepare crashes. Set parameters on the data.

In [ ]:
# --- LOAD CRASHES ---
import os

# 🔧 Make sure CRASH_CSV actually points to a real file
# (handles the common "My Drive" vs "MyDrive" issue without changing other cells)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    # If not in Colab or already mounted, this just fails quietly
    pass

if not os.path.exists(CRASH_CSV):
    alt_path = CRASH_CSV.replace("My Drive", "MyDrive")
    if os.path.exists(alt_path):
        print(f"⚠️ CRASH_CSV not found at:\n  {CRASH_CSV}")
        print(f"✅ Using alternate path instead:\n  {alt_path}")
        CRASH_CSV = alt_path
    else:
        raise FileNotFoundError(
            f"❌ Could not find crash file at:\n  {CRASH_CSV}\n"
            f"or at:\n  {alt_path}\n"
            f"→ Update CRASH_CSV to the correct path."
        )

print("\nLoading crash data...")

df = pd.read_csv(CRASH_CSV, dtype={"STREETSEGID": str}, low_memory=False)

df = df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
df["LATITUDE"]  = pd.to_numeric(df["LATITUDE"], errors="coerce")
df["LONGITUDE"] = pd.to_numeric(df["LONGITUDE"], errors="coerce")
df = df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

df["FROMDATE"] = pd.to_datetime(df["FROMDATE"], errors="coerce")
df = df[(df["FROMDATE"] >= START_DATE) & (df["FROMDATE"] <= END_DATE)].copy()

# 🔍 --- Check for duplicate CRIMEIDs before aggregation ---
if "CRIMEID" in df.columns:
    dupes = df[df.duplicated(subset=["CRIMEID"], keep=False)]
    print(f"\nDuplicate CRIMEIDs: {dupes['CRIMEID'].nunique()}")
    if not dupes.empty:
        print(dupes[["CRIMEID", "FROMDATE", "LATITUDE", "LONGITUDE"]].head(10))
    else:
        print("✅ No duplicate CRIMEIDs found.")
else:
    print("⚠️ No CRIMEID column found in dataset.")

print(f"Loaded {len(df)} crash records between {START_DATE} and {END_DATE}.")

# injury columns
fatal_cols = ["FATAL_BICYCLIST", "FATAL_DRIVER", "FATAL_PEDESTRIAN", "FATALPASSENGER", "FATALOTHER"]
major_cols = ["MAJORINJURIES_BICYCLIST", "MAJORINJURIES_DRIVER", "MAJORINJURIES_PEDESTRIAN",
              "MAJORINJURIESPASSENGER", "MAJORINJURIESOTHER"]
minor_cols = ["MINORINJURIES_BICYCLIST", "MINORINJURIES_DRIVER", "MINORINJURIES_PEDESTRIAN",
              "MINORINJURIESPASSENGER", "MINORINJURIESOTHER"]
sev_cols_all = [c for c in fatal_cols + major_cols + minor_cols if c in df.columns]

# --- Deduplicate crash records (ID-based + spatial-temporal) ---
print("\nDeduplicating crash data...")

CRASH_KEY = next((k for k in ["CRIMEID", "CRASHID", "CASE_ID", "OBJECTID", "CRASH_ID"] if k in df.columns), None)
print(f"Using crash key: {CRASH_KEY}")

if CRASH_KEY:
    pre = len(df)
    df = df.sort_values("FROMDATE").drop_duplicates(subset=[CRASH_KEY], keep="first").copy()
    print(f"Removed {pre - len(df)} duplicate {CRASH_KEY} entries (if any).")

# 2️⃣ Spatial–temporal dedupe (same spot ±15 min)
df["_lat_r"] = df["LATITUDE"].round(6)
df["_lon_r"] = df["LONGITUDE"].round(6)
df["_t15"]   = df["FROMDATE"].dt.floor("15min")

pre_dedupe = len(df)
df = (
    df.sort_values("FROMDATE")
      .drop_duplicates(subset=["_lat_r", "_lon_r", "_t15"], keep="first")
      .drop(columns=["_lat_r", "_lon_r", "_t15"])
      .copy()
)
print(f"Removed {pre_dedupe - len(df)} near-duplicate records (same location ±15 min).")
print(f"✅ {len(df)} unique crash events remain after combined deduplication.\n")

# night + known-injury filters AFTER aggregation
df["HOUR"] = df["FROMDATE"].dt.hour
df = df[df["HOUR"].isin(NIGHT_HOURS)].copy()
print(f"Filtered crashes to nighttime hours: {len(df)} records remain.")
df.drop(columns=["HOUR"], inplace=True)

# --- Exclude federal zones (Mall, Capitol, White House) ---
federal_zones = [
    # National Mall / Capitol Hill
    {"lat_min": 38.886, "lat_max": 38.895, "lon_min": -77.04, "lon_max": -76.99},
    # White House / Ellipse
    {"lat_min": 38.893, "lat_max": 38.899, "lon_min": -77.043, "lon_max": -77.032},
]

pre_len = len(df)
for z in federal_zones:
    mask = (
        (df["LATITUDE"].between(z["lat_min"], z["lat_max"])) &
        (df["LONGITUDE"].between(z["lon_min"], z["lon_max"]))
    )
    df = df[~mask]

print(f"Removed {pre_len - len(df)} crashes within approximate federal zones.")

# Optional MAR filter
if "MAR_SCORE" in df.columns:
    pre_mar_count = len(df)
    df = df[pd.to_numeric(df["MAR_SCORE"], errors="coerce") >= 100].copy()
    print(f"Applied MAR_SCORE filter >=100: {len(df)} records remain (from {pre_mar_count}).")


Exanded Severity framework

In [ ]:
# =========================
# NIGHT: Expanded Severity Framework
# Add after crash cleaning, before distance-to-light calculation
# =========================

# injury columns
fatal_cols = ["FATAL_BICYCLIST", "FATAL_DRIVER", "FATAL_PEDESTRIAN", "FATALPASSENGER", "FATALOTHER"]
major_cols = ["MAJORINJURIES_BICYCLIST", "MAJORINJURIES_DRIVER", "MAJORINJURIES_PEDESTRIAN",
              "MAJORINJURIESPASSENGER", "MAJORINJURIESOTHER"]
minor_cols = ["MINORINJURIES_BICYCLIST", "MINORINJURIES_DRIVER", "MINORINJURIES_PEDESTRIAN",
              "MINORINJURIESPASSENGER", "MINORINJURIESOTHER"]

# ensure columns exist and are numeric
for col in set(fatal_cols + major_cols + minor_cols):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    else:
        df[col] = 0

# keep only crashes with at least one injury
injury_cols_all = fatal_cols + major_cols + minor_cols
df = df[df[injury_cols_all].sum(axis=1) > 0].copy()

# row-level totals
df["TOTAL_FATALITIES"] = df[fatal_cols].sum(axis=1)
df["TOTAL_MAJOR_INJURIES"] = df[major_cols].sum(axis=1)
df["TOTAL_MINOR_INJURIES"] = df[minor_cols].sum(axis=1)

# uncapped severity
df["SEVERITY_SCORE"] = (
    7 * df["TOTAL_FATALITIES"] +
    4 * df["TOTAL_MAJOR_INJURIES"] +
    1 * df["TOTAL_MINOR_INJURIES"]
)

# capped severity
df["CAPPED_SEVERITY"] = np.where(
    df["TOTAL_FATALITIES"] > 0, 7,
    np.where(
        df["TOTAL_MAJOR_INJURIES"] > 0, 4,
        np.where(df["TOTAL_MINOR_INJURIES"] > 0, 1, 0)
    )
)

# severe/fatal flags
df["IS_SEVERE_CRASH"] = (
    (df["TOTAL_FATALITIES"] > 0) | (df["TOTAL_MAJOR_INJURIES"] > 0)
).astype(int)

df["IS_FATAL_CRASH"] = (df["TOTAL_FATALITIES"] > 0).astype(int)

print("Expanded severity framework added.")
print(df[[
    "TOTAL_FATALITIES",
    "TOTAL_MAJOR_INJURIES",
    "TOTAL_MINOR_INJURIES",
    "SEVERITY_SCORE",
    "CAPPED_SEVERITY",
    "IS_SEVERE_CRASH",
    "IS_FATAL_CRASH"
]].head())

In [ ]:
print(df[[
    "SEVERITY_SCORE",
    "CAPPED_SEVERITY",
    "IS_SEVERE_CRASH",
    "IS_FATAL_CRASH"
]].describe())

print("Fatal crashes:", df["IS_FATAL_CRASH"].sum())
print("Severe crashes:", df["IS_SEVERE_CRASH"].sum())

calculate distance to streetlight using euclidean distances, no longer haversine metrics, and balltree distancing because dealing with a large data set

In [ ]:
print(df['LATITUDE'].min(), df['LATITUDE'].max())
print(df['LONGITUDE'].min(), df['LONGITUDE'].max())
print(df[['LATITUDE','LONGITUDE']].sample(5))

In [ ]:
print("\nCalculating nearest streetlight distances for crashes...")

from pyproj import Transformer
from sklearn.neighbors import BallTree
import numpy as np

# --- Project lat/lon → UTM 18N (meters) for Euclidean distances ---
tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

# Streetlights → meters
sl_Xm, sl_Ym = tf_xy.transform(
    streetlights["LONGITUDE"].to_numpy(),
    streetlights["LATITUDE"].to_numpy()
)
SL_XY = np.column_stack([sl_Xm, sl_Ym])

# Crashes → meters
cr_Xm, cr_Ym = tf_xy.transform(
    df["LONGITUDE"].to_numpy(),
    df["LATITUDE"].to_numpy()
)
CR_XY = np.column_stack([cr_Xm, cr_Ym])

# --- Nearest neighbor using Euclidean metric in meters ---
tree = BallTree(SL_XY, metric="euclidean")
dist_m, _ = tree.query(CR_XY, k=1)
df["DIST_TO_LIGHT_M"] = dist_m.flatten()

# --- Remove crashes too close to the streetlight data boundary (in meters) ---
# (Prevents "fake dark zones" near DC–MD border where DDOT data stops)
SL_X_MIN, SL_X_MAX = sl_Xm.min(), sl_Xm.max()
SL_Y_MIN, SL_Y_MAX = sl_Ym.min(), sl_Ym.max()
BUFFER_M = 100  # ~100 meters

edge_mask_m = (
    (cr_Xm <= SL_X_MIN + BUFFER_M) |
    (cr_Xm >= SL_X_MAX - BUFFER_M) |
    (cr_Ym <= SL_Y_MIN + BUFFER_M) |
    (cr_Ym >= SL_Y_MAX - BUFFER_M)
)

pre_len = len(df)
df = df[~edge_mask_m].copy()
print(f"Removed {pre_len - len(df)} crashes near dataset boundary (within ~{BUFFER_M:.0f} m of streetlight coverage edge).")

print("\n=== DISTANCE DISTRIBUTION ===")
print(df["DIST_TO_LIGHT_M"].describe(percentiles=[0.5, 0.9, 0.99]))

print("\nCrashes > 500 m from a light:", (df["DIST_TO_LIGHT_M"] > 500).sum())
print("Crashes > 1000 m from a light:", (df["DIST_TO_LIGHT_M"] > 1000).sum())

# --- Sanity check near Capitol Hill ---
lat0, lon0 = 38.892054, -77.008611
x0, y0 = tf_xy.transform(lon0, lat0)
d_m, _ = tree.query(np.array([[x0, y0]]), k=1)
print(f"Sanity check — nearest streetlight near Capitol Hill: {d_m[0][0]:.2f} m")

# --- Summary stats ---
within = (df["DIST_TO_LIGHT_M"] <= WITHIN_THRESHOLD_M).sum()
total  = len(df)
pct_within = (within / total * 100.0) if total else 0.0
pct_far    = 100.0 - pct_within

print(f"Nighttime crashes within {WITHIN_THRESHOLD_M} m of a streetlight: {pct_within:.2f}%  ({within}/{total})")
print(f"Nighttime crashes > {WITHIN_THRESHOLD_M} m from a streetlight: {pct_far:.2f}%  ({total-within}/{total})")


Convert to UTM (meters instead of coordinates) and prepare the points that are outside of the distance threshold for clustering

In [ ]:
# Select crashes to cluster (those > FAR_THRESHOLD_M from a light) ===

print("\nSelecting crashes beyond threshold distance from streetlights...")

# Step 1: Filter crashes beyond threshold
far_df = df[df["DIST_TO_LIGHT_M"] > FAR_THRESHOLD_M].copy()
far_df.reset_index(drop=True, inplace=True)

if far_df.empty:
    print(f"No crashes beyond {FAR_THRESHOLD_M} meters. Nothing to cluster.")
else:
    print(f"Crashes > {FAR_THRESHOLD_M} m from nearest light: {len(far_df)}")

    # Step 2: Add stable position index
    far_df["ROW_POS"] = np.arange(len(far_df))

    # --- Remove duplicate locations (same lat/lon) to avoid fake dense clusters ---
    pre_dedup = len(far_df)
    far_df = far_df.drop_duplicates(subset=["LATITUDE", "LONGITUDE"]).copy()
    print(f"Removed {pre_dedup - len(far_df)} duplicate crash points before clustering.")

    # Step 3: Project lat/lon → UTM 18N (EPSG:32618)
    print("Projecting coordinates to UTM (meters)...")
    from pyproj import Transformer
    tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
    Xm, Ym = tf_xy.transform(
        far_df["LONGITUDE"].to_numpy(),
        far_df["LATITUDE"].to_numpy()
    )

    # Step 4: Store coordinates for clustering
    XY = np.column_stack([Xm, Ym])
    X_arr, Y_arr = XY[:, 0], XY[:, 1]

    print(f"Projected {len(far_df)} crash points to UTM coordinates.")

Clustering using the library complete linkage. Complete Linkage functions as so: For the actual clustering, we use a system called complete linkage. Complete linkage follow as so. It makes a clusters where the furthest distance between any two points in a cluster is not greater than the given threshold. That means that if points A, B, C and D are in a line, say 10 meters apart, and the threshold is 15 meters, the two clusters would be A,B and C,D. Not A,B,C,D. The way that it actually works is takes two points that are very close together and makes them into a cluster. It adds point after point provided that the distance between the new point and the furthest point in the cluster is less than the threshold. If it is too big, it won’t add it to that cluster and will either add it to another cluster, begin a new cluster, or it will leave it unclustered.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import numpy as np

print("\nPerforming hierarchical clustering with complete linkage...")

# --- Make sure EASTING/NORTHING exist in meters ---
# If you already have XY from a prior step:
far_df["EASTING"]  = XY[:, 0]
far_df["NORTHING"] = XY[:, 1]

# If there’s any chance of NaNs, drop them to avoid fit errors
far_xy = far_df[["EASTING", "NORTHING"]].dropna().to_numpy()
if far_xy.shape[0] == 0:
    raise ValueError("No valid points to cluster (EASTING/NORTHING are empty after dropping NaNs).")

# --- Clustering params (meters) ---
cluster_dist_threshold = 30  # meters
min_points = 3

clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=cluster_dist_threshold,
    linkage="complete",
    metric="euclidean",          # explicit (default for non-ward)
    compute_distances=False      # set True only if you plan to inspect distances_
)

# Fit clustering
labels = clustering.fit_predict(far_xy)

# Attach labels back (align lengths if we dropped NaNs)
far_df = far_df.loc[~far_df[["EASTING","NORTHING"]].isna().any(axis=1)].copy()
far_df["CLUSTER"] = labels

# --- Filter clusters by minimum size ---
counts = far_df["CLUSTER"].value_counts()
valid = counts[counts >= min_points].index
filtered_clusters = far_df[far_df["CLUSTER"].isin(valid)].copy()

print(f"Found {len(counts)} clusters, {len(valid)} with at least {min_points} points")
print(f"Total points after filtering: {len(filtered_clusters)}")

Makes the data table with following values:


Size of the cluster (COUNT)

Crash severity (SEVERITY_SUM)

Distance spread metrics (MAX_R_FROM_CENTER_M, DIAMETER_M)

Distance to streetlights

In [ ]:
# =========================
# NIGHT CLUSTER TABLE
# =========================

print("\nBuilding upgraded cluster summary table...")

fc = filtered_clusters.copy()

summaries = []

for cid in sorted(fc["CLUSTER"].unique()):
    group = fc[fc["CLUSTER"] == cid].copy()

    n_crashes = len(group)

    severity_sum = group["SEVERITY_SCORE"].sum()
    capped_sum   = group["CAPPED_SEVERITY"].sum()

    avg_severity = severity_sum / n_crashes
    avg_capped   = capped_sum / n_crashes

    severe_pct = 100 * group["IS_SEVERE_CRASH"].mean()
    fatal_pct  = 100 * group["IS_FATAL_CRASH"].mean()

    avg_lat = group["LATITUDE"].mean()
    avg_lon = group["LONGITUDE"].mean()

    summaries.append({
        "N_CRASHES": n_crashes,
        "SEVERITY_SUM": severity_sum,
        "CAPPED_SEVERITY_SUM": capped_sum,
        "AVG_SEVERITY": avg_severity,
        "AVG_CAPPED_SEVERITY": avg_capped,
        "SEVERE_CRASH_PERCENTAGE": severe_pct,
        "FATAL_CRASH_PERCENTAGE": fatal_pct,
        "AVG_LON": avg_lon,
        "AVG_LAT": avg_lat
    })

cluster_simple = pd.DataFrame(summaries)

# rank same as speeding
cluster_simple = cluster_simple.sort_values(
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

cluster_simple.insert(0, "RANK", np.arange(1, len(cluster_simple)+1))

print(f"Clusters summarized: {len(cluster_simple)}")
display(cluster_simple.head(10))


Publication ready table

In [ ]:
# =========================
# NIGHT RESULTS TABLE
# =========================

results_table = cluster_simple[
    [
        "N_CRASHES",
        "SEVERITY_SUM",
        "CAPPED_SEVERITY_SUM",
        "AVG_SEVERITY",
        "SEVERE_CRASH_PERCENTAGE",
        "FATAL_CRASH_PERCENTAGE",
        "AVG_LON",
        "AVG_LAT"
    ]
].copy()

print("Night Results Table:")
display(results_table.head(10))

Capped vs Uncapped severity

In [ ]:
# =========================
# CAPPED-SEVERITY COMPARISON
# =========================

from scipy.stats import spearmanr

# --- Capped severity ranked table ---
cluster_df_capped = cluster_simple.copy()

# Preserve original severity rank
cluster_df_capped = cluster_df_capped.rename(columns={"RANK": "ORIGINAL_RANK"})

# Re-sort by capped severity burden first, then capped severity intensity, then frequency
cluster_df_capped = cluster_df_capped.sort_values(
    ["CAPPED_SEVERITY_SUM", "AVG_CAPPED_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

# Add capped ranking
cluster_df_capped.insert(0, "CAPPED_RANK", np.arange(1, len(cluster_df_capped) + 1))

print("Capped-Severity Ranked Cluster Table:")
display(cluster_df_capped.head(10))

# --- Spearman correlation ---
rho, pval = spearmanr(
    cluster_df_capped["ORIGINAL_RANK"],
    cluster_df_capped["CAPPED_RANK"]
)

print(f"\nSpearman correlation (rho): {rho:.4f}")
print(f"P-value: {pval:.4e}")

# --- Top-10 overlap ---
k10 = min(10, len(cluster_df_capped))

top_original_10 = set(cluster_df_capped.nsmallest(k10, "ORIGINAL_RANK").index)
top_capped_10   = set(cluster_df_capped.nsmallest(k10, "CAPPED_RANK").index)

overlap_10 = len(top_original_10 & top_capped_10)
overlap_10_pct = overlap_10 / k10 * 100

print(f"\nTop-{k10} overlap: {overlap_10}/{k10} ({overlap_10_pct:.1f}%)")

# --- Top-5 overlap ---
k5 = min(5, len(cluster_df_capped))

top_original_5 = set(cluster_df_capped.nsmallest(k5, "ORIGINAL_RANK").index)
top_capped_5   = set(cluster_df_capped.nsmallest(k5, "CAPPED_RANK").index)

overlap_5 = len(top_original_5 & top_capped_5)
overlap_5_pct = overlap_5 / k5 * 100

print(f"Top-{k5} overlap: {overlap_5}/{k5} ({overlap_5_pct:.1f}%)")

Frequency Comparrisons

In [ ]:
# =========================
# FREQUENCY-RANKED TABLE
# =========================

from scipy.stats import spearmanr

cluster_df_freq = cluster_df_capped.copy()

# rename for clarity
cluster_df_freq = cluster_df_freq.rename(columns={
    "ORIGINAL_RANK": "SEVERITY_RANK",
    "CAPPED_RANK": "CAPPED_SEVERITY_RANK"
})

# sort by number of crashes
cluster_df_freq = cluster_df_freq.sort_values(
    ["N_CRASHES", "SEVERITY_SUM", "AVG_SEVERITY"],
    ascending=[False, False, False]
).reset_index(drop=True)

# add frequency rank
cluster_df_freq.insert(0, "FREQUENCY_RANK", np.arange(1, len(cluster_df_freq)+1))

print("Frequency Ranked Cluster Table:")
display(cluster_df_freq.head(10))

# =========================
# SPEARMAN CORRELATIONS
# =========================

rho_sf, p_sf = spearmanr(
    cluster_df_freq["SEVERITY_RANK"],
    cluster_df_freq["FREQUENCY_RANK"]
)

rho_cf, p_cf = spearmanr(
    cluster_df_freq["CAPPED_SEVERITY_RANK"],
    cluster_df_freq["FREQUENCY_RANK"]
)

print("\nSpearman Correlations")
print(f"Severity vs Frequency: rho = {rho_sf:.4f}, p = {p_sf:.4e}")
print(f"Capped vs Frequency:   rho = {rho_cf:.4f}, p = {p_cf:.4e}")

# =========================
# TOP-K OVERLAYS
# =========================

for k in [5, 10]:
    k_use = min(k, len(cluster_df_freq))

    top_freq = set(cluster_df_freq.nsmallest(k_use, "FREQUENCY_RANK").index)
    top_sev  = set(cluster_df_freq.nsmallest(k_use, "SEVERITY_RANK").index)
    top_cap  = set(cluster_df_freq.nsmallest(k_use, "CAPPED_SEVERITY_RANK").index)

    overlap_sf = len(top_freq & top_sev)
    overlap_cf = len(top_freq & top_cap)

    print(f"\nTop-{k_use} Overlap")
    print(f"Severity vs Frequency: {overlap_sf}/{k_use} ({100*overlap_sf/k_use:.1f}%)")
    print(f"Capped vs Frequency:   {overlap_cf}/{k_use} ({100*overlap_cf/k_use:.1f}%)")

helper for comparrisons

In [ ]:
# =========================
# EUCLIDEAN MATCHING HELPER
# Uses EPSG:32618 meters
# =========================

import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.neighbors import BallTree

tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

def add_xy(df):
    d = df.copy()
    x, y = tf_xy.transform(
        d["AVG_LON"].to_numpy(),
        d["AVG_LAT"].to_numpy()
    )
    d["X_M"] = x
    d["Y_M"] = y
    return d

Threshold Saving

In [ ]:
# =========================
# SAVE NIGHT TABLE FOR CURRENT THRESHOLD
# =========================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os

SAVE_DIR = "/content/drive/MyDrive/night_threshold_tables"
os.makedirs(SAVE_DIR, exist_ok=True)

# choose best table to save
save_df = cluster_df_freq.copy()   # most complete table

filename = f"night_{THRESHOLD_M}.csv".replace(".", "_")

path = os.path.join(SAVE_DIR, filename)

save_df.to_csv(path, index=False)

print("Saved to:", path)
print("Rows:", len(save_df))

Threshold Comparissons

In [ ]:
# =========================
# NIGHT THRESHOLD ROBUSTNESS COMPARISON
# final version
# =========================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from pyproj import Transformer
from sklearn.neighbors import BallTree

# --- project cluster centroids to EPSG:32618 for Euclidean matching ---
tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

def add_projected_xy(df):
    d = df.copy()
    x, y = tf_xy.transform(
        d["AVG_LON"].to_numpy(),
        d["AVG_LAT"].to_numpy()
    )
    d["X_M"] = x
    d["Y_M"] = y
    return d

def compare_thresholds(df_a, df_b, name_a, name_b, rank_col="SEVERITY_RANK", tolerance_m=100):
    a = add_projected_xy(df_a)
    b = add_projected_xy(df_b)

    # nearest-neighbor match in Euclidean meters
    tree = BallTree(b[["X_M", "Y_M"]].to_numpy(), metric="euclidean")
    dist_m, ind = tree.query(a[["X_M", "Y_M"]].to_numpy(), k=1)

    a["MATCH_DIST_M"] = dist_m.flatten()
    a["MATCHED_B_IDX"] = ind.flatten()

    matched = a[a["MATCH_DIST_M"] <= tolerance_m].copy()

    match_pct = 100 * len(matched) / len(a)

    print(f"\n{name_a} vs {name_b}")
    print(f"Match Percentage: {match_pct:.2f}% ({len(matched)}/{len(a)})")

    if len(matched) < 2:
        print("Spearman rho: not enough matched clusters")
        return

    rank_a = matched[rank_col].to_numpy()
    rank_b = b.iloc[matched["MATCHED_B_IDX"]][rank_col].to_numpy()

    rho, p = spearmanr(rank_a, rank_b)

    print(f"Spearman rho: {rho:.4f}")
    print(f"P-value: {p:.4e}")

# --- run the exact night threshold comparisons ---
print("Running night threshold robustness comparisons...")

compare_thresholds(night_22_5, night_30,   "22.5m", "30m")
compare_thresholds(night_30,   night_37_5, "30m",   "37.5m")
compare_thresholds(night_22_5, night_37_5, "22.5m", "37.5m")

print("\nDone.")

DBSCAN

In [ ]:
# =========================
# NIGHT DBSCAN COMPARISON
# =========================

from sklearn.cluster import DBSCAN
import pandas as pd
import numpy as np

# ---------------------------------
# DBSCAN on projected coordinates
# ---------------------------------

eps_m = 30
min_pts = 2

db = DBSCAN(eps=eps_m, min_samples=min_pts, metric="euclidean")
labels = db.fit_predict(XY)

db_df = far_df.copy()
db_df["DBSCAN_CLUSTER"] = labels

# remove noise points
db_df = db_df[db_df["DBSCAN_CLUSTER"] != -1].copy()

print("Clusters found:", db_df["DBSCAN_CLUSTER"].nunique())
print("Points kept:", len(db_df))

# ---------------------------------
# Build severity-ranked DBSCAN table
# ---------------------------------

rows = []

for cid in sorted(db_df["DBSCAN_CLUSTER"].unique()):
    g = db_df[db_df["DBSCAN_CLUSTER"] == cid].copy()

    rows.append({
        "N_CRASHES": len(g),
        "SEVERITY_SUM": g["SEVERITY_SCORE"].sum(),
        "AVG_SEVERITY": g["SEVERITY_SCORE"].mean(),
        "AVG_LON": g["LONGITUDE"].mean(),
        "AVG_LAT": g["LATITUDE"].mean()
    })

dbscan_table = pd.DataFrame(rows)

dbscan_table = dbscan_table.sort_values(
    ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

dbscan_table.insert(0, "DBSCAN_RANK", np.arange(1, len(dbscan_table)+1))

print("\nTop DBSCAN Clusters")
display(dbscan_table.head(10))

Dbscan comparrison

In [ ]:
# =========================
# DBSCAN vs PRIMARY METHOD
# TRUE TOP-K OVERLAY
# =========================

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

def compare_dbscan_to_primary(primary_df, dbscan_df, primary_rank_col="SEVERITY_RANK", db_rank_col="DBSCAN_RANK", tolerance_m=100):
    p = add_projected_xy(primary_df).copy()
    d = add_projected_xy(dbscan_df).copy()

    # nearest DBSCAN cluster for each primary cluster
    tree = BallTree(d[["X_M", "Y_M"]].to_numpy(), metric="euclidean")
    dist_m, ind = tree.query(p[["X_M", "Y_M"]].to_numpy(), k=1)

    p["MATCH_DIST_M"] = dist_m.flatten()
    p["MATCHED_DB_IDX"] = ind.flatten()

    print("\nDBSCAN vs Primary Method")

    for k in [5, 10, 20]:
        k_use = min(k, len(p), len(d))

        # top-k primary clusters
        top_primary = p.nsmallest(k_use, primary_rank_col).copy()

        # among top-k primary, keep only those with a valid DBSCAN match
        top_primary_matched = top_primary[top_primary["MATCH_DIST_M"] <= tolerance_m].copy()

        # DBSCAN clusters that those top-k primary clusters map to
        matched_db_indices_from_primary_topk = set(top_primary_matched["MATCHED_DB_IDX"].tolist())

        # actual top-k DBSCAN clusters by DBSCAN ranking
        top_dbscan = d.nsmallest(k_use, db_rank_col).copy()
        top_dbscan_indices = set(top_dbscan.index.tolist())

        # overlap = DBSCAN clusters that are both:
        # (a) matched to a top-k primary cluster
        # (b) in DBSCAN's own top-k
        overlap = len(matched_db_indices_from_primary_topk & top_dbscan_indices)

        print(f"Top-{k_use} Overlap: {overlap}/{k_use} ({100*overlap/k_use:.1f}%)")

In [ ]:
compare_dbscan_to_primary(night_30, dbscan_table)

Build cluster centroids to later connect to aadt

In [ ]:
# =========================
# NIGHT AADT STEP 1
# Build one point per cluster centroid
# =========================

import geopandas as gpd
import pandas as pd
import numpy as np
from pyproj import Transformer

# start from the full 30m ranked table
night_base = cluster_df_freq.copy()

# preserve a stable key
night_base = night_base.reset_index(drop=True)
night_base["CLUSTER_KEY"] = night_base.index

# project AVG_LON / AVG_LAT to EPSG:32618
tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)
x_m, y_m = tf_xy.transform(
    night_base["AVG_LON"].to_numpy(),
    night_base["AVG_LAT"].to_numpy()
)

night_base["X_M"] = x_m
night_base["Y_M"] = y_m

night_cluster_pts = gpd.GeoDataFrame(
    night_base,
    geometry=gpd.points_from_xy(night_base["X_M"], night_base["Y_M"]),
    crs="EPSG:32618"
)

print("Night cluster points created:", len(night_cluster_pts))

Load AADT and Match

In [ ]:
# =========================
# NIGHT AADT STEP 2
# Load AADT and match nearest segment
# =========================

import os
import zipfile
import geopandas as gpd
import pandas as pd

zip_path = "/content/drive/MyDrive/Traffic_Volume_-_2024 (3).zip"
extract_path = "/content/drive/MyDrive/aadt"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

AADT_PATH = "/content/drive/MyDrive/aadt/Traffic_Volume_-_2024.shp"

aadt = gpd.read_file(AADT_PATH)

if aadt.crs is None:
    raise ValueError("AADT shapefile has no CRS metadata.")

aadt = aadt.to_crs("EPSG:32618").copy()

AADT_COL = "AADT"

aadt_small = aadt[["geometry", AADT_COL, "ROUTEID"]].copy()
aadt_small[AADT_COL] = pd.to_numeric(aadt_small[AADT_COL], errors="coerce")
aadt_small = aadt_small[aadt_small[AADT_COL].notna() & (aadt_small[AADT_COL] > 0)].copy()
aadt_small = aadt_small.reset_index(drop=True)

night_aadt_raw = gpd.sjoin_nearest(
    night_cluster_pts,
    aadt_small,
    how="left",
    distance_col="DIST_TO_AADT_M"
).rename(columns={
    AADT_COL: "MATCHED_AADT",
    "ROUTEID": "MATCHED_ROUTEID"
})

print("Rows before join:", len(night_cluster_pts))
print("Rows after raw join:", len(night_aadt_raw))
display(night_aadt_raw.head())

compute exposure/risk

In [ ]:
# =========================
# NIGHT AADT STEP 3
# Tie-break and compute risk
# =========================

night_aadt = (
    night_aadt_raw
    .sort_values(
        by=["CLUSTER_KEY", "DIST_TO_AADT_M", "MATCHED_AADT", "MATCHED_ROUTEID"],
        ascending=[True, True, False, True]
    )
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .sort_values("CLUSTER_KEY")
    .reset_index(drop=True)
)

if len(night_aadt) != len(night_cluster_pts):
    raise ValueError("Still not one row per cluster after tie-breaking.")

# 5-year exposure to match your current 2020 to 2025 window structure
night_aadt["EXPOSURE_5YR"] = night_aadt["MATCHED_AADT"] * 365 * 5

night_aadt["RISK"] = night_aadt["SEVERITY_SUM"] / night_aadt["EXPOSURE_5YR"]
night_aadt["CAPPED_RISK"] = night_aadt["CAPPED_SEVERITY_SUM"] / night_aadt["EXPOSURE_5YR"]

night_aadt["RISK_PER_100M"] = night_aadt["RISK"] * 100_000_000
night_aadt["CAPPED_RISK_PER_100M"] = night_aadt["CAPPED_RISK"] * 100_000_000

night_risk_base = pd.DataFrame(
    night_aadt.drop(columns=["geometry", "index_right", "X_M", "Y_M"], errors="ignore")
)

print("Night risk base table created:", len(night_risk_base))
display(night_risk_base.head())

severity risk table

In [ ]:
# =========================
# NIGHT AADT STEP 4
# Risk-ranked table
# =========================

table_severity_risk = night_risk_base.sort_values(
    ["RISK_PER_100M", "SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

table_severity_risk.insert(0, "RISK_RANK", np.arange(1, len(table_severity_risk) + 1))
table_severity_risk["RANK_CHANGE"] = (
    table_severity_risk["SEVERITY_RANK"] - table_severity_risk["RISK_RANK"]
)

print("Night severity-risk ranked table:")
display(table_severity_risk[[
    "RISK_RANK",
    "SEVERITY_RANK",
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].head(10))

Revised Table

In [ ]:
# =========================
# NIGHT AADT STEP 4B
# Clean display table — separate from full risk table
# =========================

night_risk_clean_table = table_severity_risk[[
    "RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].copy()

print("Night clean severity-risk table:")
display(night_risk_clean_table.head(10))

capped severity risk

In [ ]:
# =========================
# NIGHT AADT STEP 5
# Capped-risk ranked table
# =========================

table_capped_risk = night_risk_base.sort_values(
    ["CAPPED_RISK_PER_100M", "CAPPED_SEVERITY_SUM", "N_CRASHES"],
    ascending=[False, False, False]
).reset_index(drop=True)

table_capped_risk.insert(0, "CAPPED_RISK_RANK", np.arange(1, len(table_capped_risk) + 1))
table_capped_risk["CAPPED_RANK_CHANGE"] = (
    table_capped_risk["CAPPED_SEVERITY_RANK"] - table_capped_risk["CAPPED_RISK_RANK"]
)

print("Night capped-risk ranked table:")
display(table_capped_risk[[
    "CAPPED_RISK_RANK",
    "CAPPED_SEVERITY_RANK",
    "CAPPED_RANK_CHANGE",
    "N_CRASHES",
    "SEVERITY_SUM",
    "CAPPED_SEVERITY_SUM",
    "MATCHED_AADT",
    "DIST_TO_AADT_M",
    "RISK_PER_100M",
    "CAPPED_RISK_PER_100M",
    "AVG_LON",
    "AVG_LAT"
]].head(10))

Risk comparrisons

In [ ]:
# =========================
# NIGHT AADT STEP 6
# Risk comparisons
# =========================

from scipy.stats import spearmanr

# ---- Severity vs Risk ----
rho_sr, p_sr = spearmanr(
    table_severity_risk["SEVERITY_RANK"],
    table_severity_risk["RISK_RANK"]
)

# ---- Capped Severity vs Capped Risk ----
rho_cr, p_cr = spearmanr(
    table_capped_risk["CAPPED_SEVERITY_RANK"],
    table_capped_risk["CAPPED_RISK_RANK"]
)

print("Spearman Correlations\n")
print(f"Severity vs Risk:        rho = {rho_sr:.4f}, p = {p_sr:.4e}")
print(f"Capped Severity vs Risk: rho = {rho_cr:.4f}, p = {p_cr:.4e}")

for k in [5, 10]:
    k_use = min(k, len(table_severity_risk))

    top_sev = set(table_severity_risk.nsmallest(k_use, "SEVERITY_RANK").index)
    top_risk = set(table_severity_risk.nsmallest(k_use, "RISK_RANK").index)

    overlap_sr = len(top_sev & top_risk)

    print(f"\nTop-{k_use} Severity vs Risk Overlap: {overlap_sr}/{k_use} ({100*overlap_sr/k_use:.1f}%)")

for k in [5, 10]:
    k_use = min(k, len(table_capped_risk))

    top_cap = set(table_capped_risk.nsmallest(k_use, "CAPPED_SEVERITY_RANK").index)
    top_caprisk = set(table_capped_risk.nsmallest(k_use, "CAPPED_RISK_RANK").index)

    overlap_cr = len(top_cap & top_caprisk)

    print(f"Top-{k_use} Capped Severity vs Capped Risk Overlap: {overlap_cr}/{k_use} ({100*overlap_cr/k_use:.1f}%)")

DBSCAN vs Risk Table

In [ ]:
# =====================================================
# NIGHT: DBSCAN vs AADT-NORMALIZED COMPLETE LINKAGE
# TRUE TOP-K OVERLAY ONLY
# =====================================================

night_risk_for_dbscan = table_severity_risk.copy()

compare_dbscan_to_primary(
    primary_df=night_risk_for_dbscan,
    dbscan_df=dbscan_table,
    primary_rank_col="RISK_RANK",
    db_rank_col="DBSCAN_RANK",
    tolerance_m=100
)

Load in DDOT Corridors

In [ ]:
# =========================
# NIGHT DDOT STEP 1
# Load exact same HIN source as speeding
# =========================

import geopandas as gpd

DDOT_PATH = "/content/drive/MyDrive/High_Injury_Network.geojson"

ddot = gpd.read_file(DDOT_PATH)

print("Loaded:", ddot.shape)
print("CRS:", ddot.crs)
print("Columns:")
print(ddot.columns.tolist())

display(ddot.head())

DDOT comparrison with Severity Ranked

In [ ]:
# =========================
# NIGHT vs DDOT HIGH INJURY NETWORK
# Self-contained final version
# =========================

import geopandas as gpd
import pandas as pd
import numpy as np
from pyproj import Transformer

# -------------------------
# rebuild night severity points
# -------------------------

night_base = cluster_df_freq.copy().reset_index(drop=True)
night_base["CLUSTER_KEY"] = night_base.index

tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

x_m, y_m = tf_xy.transform(
    night_base["AVG_LON"].to_numpy(),
    night_base["AVG_LAT"].to_numpy()
)

night_base["X_M"] = x_m
night_base["Y_M"] = y_m

night_pts = gpd.GeoDataFrame(
    night_base,
    geometry=gpd.points_from_xy(night_base["X_M"], night_base["Y_M"]),
    crs="EPSG:32618"
)

# -------------------------
# DDOT prep
# -------------------------

ddot = ddot.to_crs("EPSG:32618").copy()

ddot["DDOT_PRIORITY"] = np.select(
    [
        ddot["TIER_1"] == 1,
        ddot["TIER_2"] == 1,
        ddot["TIER_3"] == 1
    ],
    [1, 2, 3],
    default=99
)

# -------------------------
# nearest match
# -------------------------

night_ddot = gpd.sjoin_nearest(
    night_pts,
    ddot,
    how="left",
    distance_col="DIST_TO_HIN_M"
)

night_ddot = (
    night_ddot
    .sort_values(["SEVERITY_RANK", "DIST_TO_HIN_M"])
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .copy()
)

print("Night clusters matched:", len(night_ddot))

# -------------------------
# compare top-k
# -------------------------

for k in [5, 10, 16]:
    k_use = min(k, len(night_ddot))

    top_night = night_ddot.nsmallest(k_use, "SEVERITY_RANK").copy()

    top_ddot = ddot.nsmallest(k_use, "DDOT_PRIORITY").copy()
    top_ddot_ids = set(top_ddot.index.tolist())

    overlap = len(
        set(top_night["index_right"].dropna().astype(int)) &
        top_ddot_ids
    )

    prox = int((top_night["DIST_TO_HIN_M"] <= 100).sum())

    print(f"\nTop-{k_use} Comparison")
    print(f"Tier overlap: {overlap}/{k_use} ({100 * overlap / k_use:.1f}%)")
    print(f"Within 100m of any HIN: {prox}/{k_use} ({100 * prox / k_use:.1f}%)")

DDOT with Severity Risk Ranked

In [ ]:
# =========================
# NIGHT RISK vs DDOT HIGH INJURY NETWORK
# separate cell
# =========================

import geopandas as gpd
import numpy as np
from pyproj import Transformer

# rebuild risk-ranked cluster points from the risk table
night_risk_base = table_severity_risk.copy().reset_index(drop=True)
night_risk_base["CLUSTER_KEY"] = night_risk_base.index

tf_xy = Transformer.from_crs("EPSG:4326", "EPSG:32618", always_xy=True)

x_m, y_m = tf_xy.transform(
    night_risk_base["AVG_LON"].to_numpy(),
    night_risk_base["AVG_LAT"].to_numpy()
)

night_risk_base["X_M"] = x_m
night_risk_base["Y_M"] = y_m

night_risk_pts = gpd.GeoDataFrame(
    night_risk_base,
    geometry=gpd.points_from_xy(night_risk_base["X_M"], night_risk_base["Y_M"]),
    crs="EPSG:32618"
)

# ensure DDOT priority exists
if "DDOT_PRIORITY" not in ddot.columns:
    ddot = ddot.to_crs("EPSG:32618").copy()
    ddot["DDOT_PRIORITY"] = np.select(
        [
            ddot["TIER_1"] == 1,
            ddot["TIER_2"] == 1,
            ddot["TIER_3"] == 1
        ],
        [1, 2, 3],
        default=99
    )

# nearest HIN match
night_risk_ddot = gpd.sjoin_nearest(
    night_risk_pts,
    ddot,
    how="left",
    distance_col="DIST_TO_HIN_M"
)

night_risk_ddot = (
    night_risk_ddot
    .sort_values(["RISK_RANK", "DIST_TO_HIN_M"])
    .drop_duplicates(subset="CLUSTER_KEY", keep="first")
    .copy()
)

print("Night risk clusters matched:", len(night_risk_ddot))

# top-k comparisons
for k in [5, 10, 16]:
    k_use = min(k, len(night_risk_ddot))

    top_night = night_risk_ddot.nsmallest(k_use, "RISK_RANK").copy()
    top_ddot = ddot.nsmallest(k_use, "DDOT_PRIORITY").copy()

    top_ddot_ids = set(top_ddot.index.tolist())

    overlap = len(
        set(top_night["index_right"].dropna().astype(int)) &
        top_ddot_ids
    )

    prox = int((top_night["DIST_TO_HIN_M"] <= 100).sum())

    print(f"\nTop-{k_use} Comparison")
    print(f"Tier overlap: {overlap}/{k_use} ({100 * overlap / k_use:.1f}%)")
    print(f"Within 100m of any HIN: {prox}/{k_use} ({100 * prox / k_use:.1f}%)")

Mapping

In [ ]:
# =========================
# NIGHT MAP
# Streetlights + far-from-light crashes + TOP 10 clusters by highest severity sum
# =========================

import folium
from branca.colormap import linear
import numpy as np

print("\nMapping streetlights + far-from-light crashes + TOP 10 clusters by SEVERITY_SUM...")

# -----------------------------
# 0) Prep far-from-light crash points
# -----------------------------
fc_pts = fc[["LATITUDE", "LONGITUDE"]].copy()
fc_pts["LATITUDE"]  = pd.to_numeric(fc_pts["LATITUDE"], errors="coerce")
fc_pts["LONGITUDE"] = pd.to_numeric(fc_pts["LONGITUDE"], errors="coerce")

n_total = len(fc_pts)
fc_pts = fc_pts.dropna(subset=["LATITUDE", "LONGITUDE"])
n_valid = len(fc_pts)

print(f"Far-from-light crashes total: {n_total:,}")
print(f"Far-from-light crashes with valid coords plotted: {n_valid:,}")

# -----------------------------
# 1) Map init
# -----------------------------
m = folium.Map(
    location=[38.9072, -77.0369],
    zoom_start=12,
    tiles="cartodbpositron",
    prefer_canvas=True
)

m.fit_bounds([[LAT_MIN, LON_MIN], [LAT_MAX, LON_MAX]])

# -----------------------------
# 2) Streetlights base layer
# -----------------------------
fg_lights = folium.FeatureGroup(name="Streetlights", show=True)

for _, row in streetlights.iterrows():
    folium.CircleMarker(
        location=[float(row["LATITUDE"]), float(row["LONGITUDE"])],
        radius=1.2,
        color="#0d6efd",
        fill=True,
        fill_opacity=0.25,
        weight=0
    ).add_to(fg_lights)

fg_lights.add_to(m)

# -----------------------------
# 3) Far-from-light crashes
# -----------------------------
fg_fc = folium.FeatureGroup(name="Far-from-light crashes", show=True)

for lat, lon in fc_pts[["LATITUDE", "LONGITUDE"]].to_numpy():
    folium.CircleMarker(
        location=[float(lat), float(lon)],
        radius=3.0,
        stroke=False,
        fill=True,
        fill_color="#3388ff",
        fill_opacity=0.6
    ).add_to(fg_fc)

fg_fc.add_to(m)

# -----------------------------
# 4) Top 10 clusters by highest severity sum
# -----------------------------
top_clusters = (
    cluster_simple
    .sort_values(
        ["SEVERITY_SUM", "AVG_SEVERITY", "N_CRASHES"],
        ascending=[False, False, False]
    )
    .head(10)
    .copy()
)

if top_clusters.empty:
    print("No clusters available to map.")
else:
    vmin_cl = float(top_clusters["SEVERITY_SUM"].min())
    vmax_cl = float(top_clusters["SEVERITY_SUM"].max())

    if vmin_cl == vmax_cl:
        vmin_cl = 0.0

    cmap_cl = linear.Reds_09.scale(vmin_cl, vmax_cl)

    fg_cl = folium.FeatureGroup(
        name="Top 10 clusters by severity sum",
        show=True
    )

    for _, row in top_clusters.iterrows():
        rank = int(row["RANK"])
        sev_sum = float(row["SEVERITY_SUM"])
        n_crashes = int(row["N_CRASHES"])
        avg_sev = float(row["AVG_SEVERITY"])
        severe_pct = float(row["SEVERE_CRASH_PERCENTAGE"])
        fatal_pct = float(row["FATAL_CRASH_PERCENTAGE"])

        if vmax_cl == vmin_cl:
            radius = 15.0
        else:
            radius = 10.0 + 25.0 * (sev_sum - vmin_cl) / (vmax_cl - vmin_cl)

        color = cmap_cl(sev_sum)

        popup = (
            f"<b>Severity Rank:</b> {rank}<br>"
            f"<b>Crashes:</b> {n_crashes}<br>"
            f"<b>Severity Sum:</b> {int(sev_sum)}<br>"
            f"<b>Average Severity:</b> {avg_sev:.2f}<br>"
            f"<b>Severe Crash %:</b> {severe_pct:.1f}%<br>"
            f"<b>Fatal Crash %:</b> {fatal_pct:.1f}%<br>"
            f"<b>Longitude:</b> {float(row['AVG_LON']):.6f}<br>"
            f"<b>Latitude:</b> {float(row['AVG_LAT']):.6f}"
        )

        tooltip = (
            f"Rank {rank} | "
            f"Severity Sum {int(sev_sum)} | "
            f"{n_crashes} crashes"
        )

        folium.CircleMarker(
            location=[float(row["AVG_LAT"]), float(row["AVG_LON"])],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.9,
            weight=2,
            popup=folium.Popup(popup, max_width=380),
            tooltip=tooltip
        ).add_to(fg_cl)

    fg_cl.add_to(m)

    cmap_cl.caption = "Top 10 cluster severity sum"
    cmap_cl.add_to(m)

# -----------------------------
# 5) Layer control
# -----------------------------
folium.LayerControl(collapsed=False).add_to(m)

print("✅ Map ready: streetlights + far-from-light crashes + top 10 clusters ranked by highest SEVERITY_SUM.")
m